In [3]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import warnings
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')

In [5]:
def extract_pos_coordinates(filepath, columns_to_extract=None, output_csv=None):
    """
    Extract specified columns from a position file.
    
    Parameters:
    - filepath: Path to the input file
    - columns_to_extract: List of column indices to extract (0-based). 
                         If None, extracts first 4 columns by default.
    - output_csv: Optional path to save results as CSV
    """
    #if columns_to_extract is None:
        #columns_to_extract = [0, 1, 2, 3,5,6,7,8,9,10,11]# Default: first 4 columns
    
    extracted_rows = []
    column_names = []

    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            # Skip metadata and comments
            if not line or line.startswith('%'):
                continue
            
            # Extract and clean the comma-separated values
            parts = [p.strip() for p in line.split(',')]
            
            # Extract only the specified columns
            try:
                extracted_values = []
                for col_idx in columns_to_extract:
                    if col_idx < len(parts):
                        # Try to convert to float, otherwise keep as string
                        try:
                            extracted_values.append(float(parts[col_idx]))
                        except ValueError:
                            extracted_values.append(parts[col_idx])
                    else:
                        extracted_values.append(None)  # or '' for empty string
                
                extracted_rows.append(extracted_values)
            except (IndexError, ValueError) as e:
                print(f"Warning: Skipping line due to error: {e}")
                print(f"Line content: {line}")
                continue

    # Optional: write to CSV
    if output_csv:
        import csv
        # Generate column headers based on indices
        column_names = [f'Column_{idx}' for idx in columns_to_extract]
        
        with open(output_csv, 'w', newline='') as f_out:
            writer = csv.writer(f_out)
            writer.writerow(column_names)
            writer.writerows(extracted_rows)

    return extracted_rows

# Example usage
pos_file_path = r'F:\zizo\RTKCorrection\src\research\research_data\testBaseIONOGPSpos'



# Example 3: Extract with CSV output
output1 = extract_pos_coordinates(
    pos_file_path, 
    columns_to_extract=[0, 1, 2, 3],  # GPST, Latitude, Longitude
    output_csv='extracted_coordinates_yGPS.csv'
)

pd.set_option('display.precision', 10)
df_x = pd.DataFrame(output1, columns=['GPST', 'x_y', 'y_y', 'z_y'])


In [15]:
df_x

,GPST,x_y,y_y,z_y,ns,sdx(m),sdy(m),sdz(m),sdxy(m),sdyz(m),sdzx(m)
0,2024/02/14 04:19:28.00,461332.1582,5.6344514771e+06,2.9459406828e+06,13.0,3.5287,11.2818,6.9751,0.4251,5.9601,0.7971
1,2024/02/14 04:19:29.00,461332.0927,5.6344515642e+06,2.9459409229e+06,13.0,3.5287,11.2825,6.9752,0.4270,5.9598,0.7981
2,2024/02/14 04:19:30.00,461332.1570,5.6344516425e+06,2.9459407543e+06,13.0,3.5287,11.2832,6.9753,0.4289,5.9595,0.7990
3,2024/02/14 04:19:31.00,461332.1452,5.6344513007e+06,2.9459406677e+06,13.0,3.5287,11.2839,6.9755,0.4308,5.9593,0.8000
4,2024/02/14 04:19:32.00,461332.2278,5.6344510727e+06,2.9459404919e+06,13.0,3.5287,11.2846,6.9756,0.4326,5.9590,0.8009
...,...,...,...,...,...,...,...,...,...,...,...
7968,2024/02/14 06:32:20.01,464124.0612,5.6326319059e+06,2.9490136174e+06,12.0,6.0030,21.8619,8.2872,-5.2398,12.2391,-2.5294
7969,2024/02/14 06:32:21.01,464124.1856,5.6326333853e+06,2.9490157864e+06,12.0,6.0032,21.8594,8.2869,-5.2407,12.2383,-2.5301
7970,2024/02/14 06:32:22.01,464124.7210,5.6326347517e+06,2.9490175730e+06,11.0,10.7966,30.3993,11.5897,-14.7332,17.9150,-8.8946
7971,2024/02/14 06:32:23.01,464132.2236,5.6326295838e+06,2.9490205933e+06,11.0,10.7976,30.3961,11.5892,-14.7335,17.9138,-8.8950


In [16]:
df_x.describe()

,x_y,y_y,z_y,ns,sdx(m),sdy(m),sdz(m),sdxy(m),sdyz(m),sdzx(m)
count,7973.0000000000,7.9730000000e+03,7.9730000000e+03,7973.0000000000,7973.0000000000,7973.0000000000,7973.0000000000,7973.0000000000,7973.0000000000,7973.0000000000
mean,460282.8545129061,5.6337351325e+06,2.9474732216e+06,12.0110372507,5.3980009407,20.2717080020,7.8993545466,1.8899694845,9.7368900916,1.6638900665
std,1663.8106432953,1.0611374776e+03,2.0544020083e+03,0.5409740061,1.3640609926,5.8129304562,1.2225339416,6.0940965399,3.5199131855,3.3311321573
min,457728.2899000000,5.6316549269e+06,2.9436784246e+06,7.0000000000,3.5237000000,11.2818000000,6.1523000000,-29.7327000000,-3.4254000000,-21.2682000000
25%,458612.7612000000,5.6329811817e+06,2.9459398613e+06,12.0000000000,5.0139000000,16.8641000000,7.0045000000,-4.0625000000,7.0318000000,-0.9810000000
50%,460150.6952000000,5.6334910490e+06,2.9475517752e+06,12.0000000000,5.4657000000,18.8325000000,7.8529000000,5.9891000000,8.8179000000,3.8130000000
75%,461474.9851000000,5.6345679003e+06,2.9490750671e+06,12.0000000000,5.6691000000,24.6884000000,8.5248000000,6.9030000000,12.9835000000,3.9674000000
max,464241.0758000000,5.6356983371e+06,2.9516372180e+06,13.0000000000,21.7328000000,53.0653000000,24.6342000000,25.9501000000,35.1415000000,17.3102000000


In [7]:
# --- 1. DATA LOADING AND UNIFICATION ---
def load_and_unify_data(spp_filepath, kinematic_filepath):
    """Loads SPP and Kinematic data, aligns them, calculates differences, and returns unified DataFrame."""
    try:
        spp_df = pd.read_csv(spp_filepath)
        kin_df = pd.read_csv(kinematic_filepath)
        print(f"✓ SPP file loaded with {len(spp_df)} rows.")
        print(f"✓ Kinematic file loaded with {len(kin_df)} rows.")
    except Exception as e:
        print(f"✗ Error loading files: {e}")
        return None, None, None, None, None

    # Align both datasets to same size
    min_len = min(len(spp_df), len(kin_df))
    spp_df = spp_df.iloc[:min_len].copy()
    kin_df = kin_df.iloc[:min_len].copy()
    print(f"✓ Data aligned to {min_len} rows.")

    # Explicit columns
    spp_x_col, spp_y_col, spp_z_col = 'x_x', 'y_x', 'z_x'
    kin_x_col, kin_y_col, kin_z_col = 'x_y', 'y_y', 'z_y'

    

    # Merge SPP + Kinematic
    unified_df = spp_df.copy()
    unified_df['kin_x'] = kin_df[kin_x_col]
    unified_df['kin_y'] = kin_df[kin_y_col]
    unified_df['kin_z'] = kin_df[kin_z_col]

    unified_df['diff_x'] = unified_df['kin_x'] - unified_df[spp_x_col]
    unified_df['diff_y'] = unified_df['kin_y'] - unified_df[spp_y_col]
    unified_df['diff_z'] = unified_df['kin_z'] - unified_df[spp_z_col]
    """
    unified_df['GPST'] = pd.to_datetime(kin_df['GPST'], format='%Y/%m/%d %H:%M:%S.%f')
    unified_df['time_elapsed'] = (unified_df['GPST'] - unified_df['GPST'].iloc[0]).dt.total_seconds()

    # Compute speed = distance / time
    unified_df['dx'] = unified_df['x_x'].diff()
    unified_df['dy'] = unified_df['y_x'].diff()
    unified_df['dz'] = unified_df['z_x'].diff()
    unified_df['dt'] = unified_df['time_elapsed'].diff()
    unified_df['v_x'] = unified_df['dx'] / unified_df['dt']
    unified_df['v_y'] = unified_df['dy'] / unified_df['dt']
    unified_df['v_z'] = unified_df['dz'] / unified_df['dt']
    unified_df['distance'] = np.sqrt(unified_df['dx']**2 + unified_df['dy']**2 + unified_df['dz']**2)
    unified_df['speed'] = unified_df['distance'] / unified_df['dt']

        # Calculate the 3D magnitude of the standard deviation vector
    unified_df['sd_magnitude'] = np.sqrt(
        spp_df['sdx(m)']**2 + spp_df['sdy(m)']**2 + spp_df['sdz(m)']**2
    )

    # Interaction between number of satellites and uncertainty
    unified_df['ns_x_sd_magnitude'] = spp_df['ns'] * unified_df['sd_magnitude']

    # Ratios to capture geometric effects (prevent division by zero)
    unified_df['sdx_sdy_ratio'] = spp_df['sdx(m)'] / (spp_df['sdy(m)'] + 1e-6)
    unified_df['sdy_sdz_ratio'] = spp_df['sdy(m)'] / (spp_df['sdz(m)'] + 1e-6)
    unified_df=unified_df.drop(columns=['dx','dy','dz','distance','dt','time_elapsed','GPST','sdxy(m)','sdyz(m)','sdzx(m)'],axis = 1)

    #unified_df['x_lag1'] = unified_df['x_x'].shift(1)
    #unified_df['y_lag1'] = unified_df['y_x'].shift(1)
    #unified_df['z_lag1'] = unified_df['z_x'].shift(1)
    #unified_df = unified_df.drop(index=0)"""

    feature_cols = ['x_x', 'y_x', 'z_x', 'ns', 'sdx(m)', 'sdy(m)', 'sdz(m)']

    return unified_df, feature_cols, spp_x_col, spp_y_col, spp_z_col

spp_filepath = r'F:\zizo\RTKCorrection\src\research\extracted_coordinates_xGPS.csv'
kinematic_filepath = r'F:\zizo\RTKCorrection\src\research\extracted_coordinates_yGPS.csv'

unified_df, feature_cols, spp_x, spp_y, spp_z = load_and_unify_data(
    spp_filepath, kinematic_filepath
)



✓ SPP file loaded with 7881 rows.
✓ Kinematic file loaded with 7526 rows.
✓ Data aligned to 7526 rows.


In [8]:
unified_df.tail().T

,7521,7522,7523,7524,7525
GPST,2024/02/14 06:26:23.00,2024/02/14 06:26:24.00,2024/02/14 06:26:26.00,2024/02/14 06:26:27.00,2024/02/14 06:26:28.00
x_x,463025.5186,463027.5898,463031.6,463038.3771,463038.8097
y_x,5633285.5103000002,5633283.6021999996,5633278.7386999996,5633286.6002000002,5633278.2479999997
z_x,2947857.8289999999,2947858.4774000002,2947867.5682999999,2947855.9611999998,2947869.8757000002
ns,7.0,7.0,8.0,7.0,8.0
sdx(m),4.3334,4.333,4.0141,4.3318,4.0138
sdy(m),14.1138,14.1103,13.8008,14.0999,13.7965
sdz(m),9.8189,9.8218,6.9713,9.8302,6.9703
sdxy(m),5.0656,5.0631,4.5661,5.0554,4.5632
sdyz(m),6.4287,6.4297,7.8418,6.4329,7.8394


In [10]:
unified_df['diff_x'].describe()

count    7526.0000000000
mean      132.1307912304
std       361.6035788419
min      -694.1736000000
25%       -44.6470500000
50%        -1.1991500000
75%       324.8857750000
max      1240.9043000000
Name: diff_x, dtype: float64

In [ ]:
# --- 3. EVALUATION METRICS ---
def evaluate_model(model_name, y_test_vals, y_pred):
    print("\n" + "=" * 80 + f"\nFINAL EVALUATION: {model_name}\n" + "=" * 80)
    per_axis_rmse = []
    per_axis_r2 = []

    for i, axis in enumerate(['X', 'Y', 'Z']): # i is for index, axis is the string
        rmse = np.sqrt(mean_squared_error(y_test_vals[:, i], y_pred[:, i])) # y_pred -> -2.225, y_test_vals -> -2.3353
        r2 = r2_score(y_test_vals[:, i], y_pred[:, i])
        per_axis_rmse.append(rmse)
        per_axis_r2.append(r2)
        print(f"--- Axis {axis} ---\n  R² Score: {r2:.4f}\n  RMSE (m): {rmse:.4f}\n")

    overall_rmse = np.sqrt(np.sum(np.square(per_axis_rmse)))
    print(f"--- Overall 3D Error ---\n  Euclidean RMSE (m): {overall_rmse:.4f}")

    return {
        'r2_scores': per_axis_r2,
        'rmse_per_axis': per_axis_rmse,
        'overall_rmse': overall_rmse
    }


In [20]:
# --- 2. MODEL TRAINING AND EVALUATION ---
def train_and_evaluate(unified_df, feature_cols):
    """Trains and evaluates both RandomForest and XGBoost models."""
    X = unified_df[feature_cols]
    y = unified_df[['diff_x', 'diff_y', 'diff_z']]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # --- Model configurations ---
    models = {
        'RandomForest': (RandomForestRegressor(random_state=42), {
            'n_estimators': [100, 200],
            'max_depth': [10, 20],
            'min_samples_leaf': [2, 4]
        })
    }

    results = {}
    print("\n" + "=" * 80 + "\nMODEL TRAINING AND COMPARISON\n" + "=" * 80)

    for name, (model, params) in models.items():
        print(f"\n🔹 Tuning {name}...")
        grid = GridSearchCV(model, params, cv=5, scoring='r2', n_jobs=-1, verbose=1)
        grid.fit(X_train_scaled, y_train)
        best_model = grid.best_estimator_
        print(f"  ✓ Best Params: {grid.best_params_}")
        print(f"  ✓ Best CV R² Score: {grid.best_score_:.4f}")
        # y_pred is numpy array
        y_pred = best_model.predict(X_test_scaled)
        print(y_pred)
        # y_test.values is used to convert DataFrame to numpy array
        model_results = evaluate_model(name, y_test.values, y_pred)
        #plot_results(y_test.values, y_pred, name)

        results[name] = {
            'model': best_model,
            'y_pred': y_pred,
            'metrics': model_results
        }

    return results, scaler, X_test

results, scaler, X_test = train_and_evaluate(unified_df, feature_cols)
test_df_x = unified_df.loc[X_test.index]



MODEL TRAINING AND COMPARISON

🔹 Tuning RandomForest...
Fitting 5 folds for each of 8 candidates, totalling 40 fits
  ✓ Best Params: {'max_depth': 20, 'min_samples_leaf': 2, 'n_estimators': 200}
  ✓ Best CV R² Score: 0.9903
[[ -20.19439306  -15.9053469     1.28841018]
 [  98.99519257   28.08279325 -110.59321189]
 [  -9.22369131  -22.47557711   13.6933863 ]
 ...
 [ -26.25156003   24.53848585  -71.43789028]
 [  -1.59275194   -8.37234243   -7.37071786]
 [ -67.419687   -186.83578309  298.61313651]]

FINAL EVALUATION: RandomForest
--- Axis X ---
  R² Score: 0.9920
  RMSE (m): 7.3787

--- Axis Y ---
  R² Score: 0.9875
  RMSE (m): 5.3249

--- Axis Z ---
  R² Score: 0.9964
  RMSE (m): 5.2942

--- Overall 3D Error ---
  Euclidean RMSE (m): 10.5275


In [ ]:
# --- 4. ACCURACY IMPROVEMENT ---
def calculate_accuracy_improvement(test_df, y_pred_corrected, spp_cols):
    spp_x_col, spp_y_col, spp_z_col = spp_cols
    orig_error = np.sqrt(
        test_df['diff_x']**2 + test_df['diff_y']**2 + test_df['diff_z']**2
    ).mean()
    # y_pred_corrected came from predict(X_test_scaled)
    corrected_x = test_df[spp_x_col] + y_pred_corrected[:, 0]
    corrected_y = test_df[spp_y_col] + y_pred_corrected[:, 1]
    corrected_z = test_df[spp_z_col] + y_pred_corrected[:, 2]
    # corrected_error ---> kinematic - AI_model_prediction
    corrected_error = np.sqrt(
        (test_df['kin_x'] - corrected_x)**2 +
        (test_df['kin_y'] - corrected_y)**2 +
        (test_df['kin_z'] - corrected_z)**2
    ).mean()

    improvement = orig_error - corrected_error
    percent = (improvement / orig_error) * 100 if orig_error != 0 else 0

    print(f"\n📊 3D Accuracy Improvement:")
    print(f"   Original Mean Error : {orig_error:.4f} m")
    print(f"   Corrected Mean Error: {corrected_error:.4f} m")
    print(f"   Improvement         : {improvement:.4f} m ({percent:.2f}%)")

for model_name, info in results.items():
    print("\n" + "=" * 80)
    print(f"🔍 ACCURACY IMPROVEMENT ANALYSIS FOR {model_name}")
    # test_df_x is x_test , <----------------> info['y_pred'] is for values of y_pred, <----------------> spp_x_col, spp_y_col, spp_z_col = 'x_x', 'y_x', 'z_x'
    calculate_accuracy_improvement(test_df_x, info['y_pred'], (spp_x, spp_y, spp_z))

        # Save model
    model_filename = f"{model_name.lower()}_gnss_model.joblib"
    joblib.dump(info['model'], model_filename)
    print(f"✓ {model_name} model saved as {model_filename}")


🔍 ACCURACY IMPROVEMENT ANALYSIS FOR RandomForest

📊 3D Accuracy Improvement:
   Original Mean Error : 85.9293 m
   Corrected Mean Error: 5.5366 m
   Improvement         : 80.3927 m (93.56%)
✓ RandomForest model saved as randomforest_gnss_model.joblib


In [33]:

# --- 5. PLOTTING ---
def plot_results(y_test, y_pred, model_name):
    fig, axes = plt.subplots(3, 1, figsize=(8, 12), tight_layout=True)
    fig.suptitle(f'{model_name} Performance: Predicted vs Actual Error', fontsize=16, y=1.03)
    for i, label in enumerate(['X', 'Y', 'Z']):
        ax = axes[i]
        ax.scatter(y_test[:, i], y_pred[:, i], alpha=0.6, s=20, edgecolors='k', c='skyblue')
        ax.plot([y_test[:, i].min(), y_test[:, i].max()],
                [y_test[:, i].min(), y_test[:, i].max()], 'r--', lw=2)
        ax.set_xlabel(f'Actual Error on Axis {label} (m)')
        ax.set_ylabel(f'Predicted Error on Axis {label} (m)')
        ax.set_title(f'Axis {label}')
        ax.grid(True, linestyle='--', alpha=0.6)
    plt.savefig(f'{model_name}_performance.png', dpi=300)
    print(f"✓ Performance plot saved: {model_name}_performance.png")

